# 07 - Model comparison, round 2: richer feature set

Rerun the same four models from `05_model_comparison.ipynb`, now on the 34-feature set from `06_rich_feature_engineering.ipynb` (24 original + 10 new: pawn structure, king safety, development, tactical exposure). `king_safety_diff` (0.21) and `passed_pawns_diff` (0.19) correlated with `white_win` more strongly than anything except `material_diff` and `mobility_diff` — this is the real test of whether that translates into better predictions.

In [1]:
import sys
sys.path.append('..')

import time
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.inspection import permutation_importance

from src.features import extract_features_v2

train_df = pd.read_csv('../data/processed/train_features_v2.csv', low_memory=False)
test_df = pd.read_csv('../data/processed/test_features_v2.csv', low_memory=False)

feature_cols = list(extract_features_v2(train_df['fen'].iloc[0]).keys())
X_train, y_train = train_df[feature_cols], train_df['white_win']
X_test, y_test = test_df[feature_cols], test_df['white_win']

print(f'{len(feature_cols)} features')
X_train.shape, X_test.shape

34 features


((822936, 34), (208378, 34))

In [2]:
def evaluate_model(name, model, X_tr, X_te):
    start = time.time()
    model.fit(X_tr, y_train)
    fit_seconds = time.time() - start

    preds = model.predict(X_te)
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'recall_black_win': recall_score(y_test, preds, pos_label=0),
        'fit_seconds': fit_seconds,
    }

    print(f'{name}  (fit in {fit_seconds:.1f}s)')
    print(classification_report(y_test, preds, target_names=['black_win', 'white_win']))

    return metrics, model


results = []

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

metrics, _ = evaluate_model('Logistic Regression', LogisticRegression(max_iter=1000), X_train_scaled, X_test_scaled)
results.append(metrics)

Logistic Regression  (fit in 0.6s)
              precision    recall  f1-score   support

   black_win       0.67      0.58      0.62    101264
   white_win       0.64      0.73      0.68    107114

    accuracy                           0.65    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.65      0.65      0.65    208378



In [3]:
tree_limited = DecisionTreeClassifier(max_depth=8, random_state=42)
metrics, tree_limited = evaluate_model('Decision Tree (max_depth=8)', tree_limited, X_train, X_test)
results.append(metrics)

rf = RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
metrics, rf = evaluate_model('Random Forest', rf, X_train, X_test)
results.append(metrics)

hgb = HistGradientBoostingClassifier(max_iter=300, random_state=42)
metrics, hgb = evaluate_model('Gradient Boosting (HistGB)', hgb, X_train, X_test)
results.append(metrics)

Decision Tree (max_depth=8)  (fit in 1.8s)
              precision    recall  f1-score   support

   black_win       0.68      0.53      0.60    101264
   white_win       0.63      0.76      0.69    107114

    accuracy                           0.65    208378
   macro avg       0.65      0.65      0.64    208378
weighted avg       0.65      0.65      0.64    208378



Random Forest  (fit in 14.4s)
              precision    recall  f1-score   support

   black_win       0.69      0.53      0.60    101264
   white_win       0.64      0.77      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.66      0.65    208378



Gradient Boosting (HistGB)  (fit in 8.7s)
              precision    recall  f1-score   support

   black_win       0.68      0.56      0.62    101264
   white_win       0.65      0.75      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.66      0.66    208378
weighted avg       0.66      0.66      0.66    208378



## Before/after: original 24 features vs. richer 34 features

In [4]:
# round-1 numbers from 05_model_comparison.ipynb, hardcoded here for a direct comparison
round1 = pd.DataFrame([
    {'model': 'Logistic Regression', 'accuracy': 0.6493, 'f1': 0.6834, 'recall_black_win': 0.5573},
    {'model': 'Decision Tree (max_depth=8)', 'accuracy': 0.6479, 'f1': 0.6966, 'recall_black_win': 0.5015},
    {'model': 'Random Forest', 'accuracy': 0.6532, 'f1': 0.6965, 'recall_black_win': 0.5254},
    {'model': 'Gradient Boosting (HistGB)', 'accuracy': 0.6537, 'f1': 0.6917, 'recall_black_win': 0.5460},
]).set_index('model')

round2 = pd.DataFrame(results).set_index('model')[['accuracy', 'f1', 'recall_black_win']]

comparison = round1.join(round2, lsuffix='_v1_24feat', rsuffix='_v2_34feat')
comparison[[
    'accuracy_v1_24feat', 'accuracy_v2_34feat',
    'f1_v1_24feat', 'f1_v2_34feat',
    'recall_black_win_v1_24feat', 'recall_black_win_v2_34feat',
]].round(4)

,accuracy_v1_24feat,accuracy_v2_34feat,f1_v1_24feat,f1_v2_34feat,recall_black_win_v1_24feat,recall_black_win_v2_34feat
model,,,,,,
Logistic Regression,0.6493,0.6536,0.6834,0.6831,0.5573,0.5768
Decision Tree (max_depth=8),0.6479,0.6494,0.6966,0.6904,0.5015,0.5316
Random Forest,0.6532,0.6578,0.6965,0.6995,0.5254,0.5343
Gradient Boosting (HistGB),0.6537,0.6602,0.6917,0.6951,0.5460,0.5613


## Where do the new features rank?
Random forest importances across the full 34-feature set.

In [5]:
pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False).head(15)

,feature,importance
10,material_diff,0.361294
18,mobility_diff,0.126162
27,passed_pawns_diff,0.055679
16,mobility_white,0.052393
17,mobility_black,0.051821
28,king_safety_diff,0.041877
0,white_pawns,0.032612
5,black_pawns,0.028617
23,fullmove_number,0.023534
9,black_queens,0.019695


## Experiment: drop the lowest-importance features
Get the full importance ranking (not just top 15), identify the weakest features, and retrain with them dropped to see if pruning noise helps.

In [6]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

print(importance_df.to_string(index=False))

# drop the bottom third by importance
n_drop = len(feature_cols) // 3
weak_features = importance_df.tail(n_drop)['feature'].tolist()
print(f'\ndropping {n_drop} weakest features:', weak_features)

pruned_feature_cols = [c for c in feature_cols if c not in weak_features]

                feature  importance
          material_diff    0.361294
          mobility_diff    0.126162
      passed_pawns_diff    0.055679
         mobility_white    0.052393
         mobility_black    0.051821
       king_safety_diff    0.041877
            white_pawns    0.032612
            black_pawns    0.028617
        fullmove_number    0.023534
           black_queens    0.019695
           white_queens    0.018726
            white_rooks    0.018004
            black_rooks    0.014839
    isolated_pawns_diff    0.013794
    hanging_pieces_diff    0.012638
       pawn_shield_diff    0.012507
    center_control_diff    0.012021
          black_bishops    0.011563
    advanced_pawns_diff    0.011082
          white_bishops    0.010442
          white_knights    0.009614
   white_center_control    0.007684
   black_center_control    0.007468
          black_knights    0.007179
undeveloped_minors_diff    0.006753
   rook_open_files_diff    0.006747
     doubled_pawns_diff    0

In [7]:
pruning_results = []

scaler_pruned = StandardScaler().fit(train_df[pruned_feature_cols])
X_train_pruned_scaled = scaler_pruned.transform(train_df[pruned_feature_cols])
X_test_pruned_scaled = scaler_pruned.transform(test_df[pruned_feature_cols])

metrics, _ = evaluate_model(
    'Logistic Regression (pruned)', LogisticRegression(max_iter=1000),
    X_train_pruned_scaled, X_test_pruned_scaled,
)
pruning_results.append(metrics)

metrics, _ = evaluate_model(
    'Random Forest (pruned)', RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
    train_df[pruned_feature_cols], test_df[pruned_feature_cols],
)
pruning_results.append(metrics)

metrics, _ = evaluate_model(
    'Gradient Boosting (pruned)', HistGradientBoostingClassifier(max_iter=300, random_state=42),
    train_df[pruned_feature_cols], test_df[pruned_feature_cols],
)
pruning_results.append(metrics)

Logistic Regression (pruned)  (fit in 0.5s)
              precision    recall  f1-score   support

   black_win       0.67      0.55      0.61    101264
   white_win       0.64      0.74      0.69    107114

    accuracy                           0.65    208378
   macro avg       0.65      0.65      0.65    208378
weighted avg       0.65      0.65      0.65    208378



Random Forest (pruned)  (fit in 14.1s)
              precision    recall  f1-score   support

   black_win       0.69      0.53      0.60    101264
   white_win       0.64      0.78      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.66      0.65    208378



Gradient Boosting (pruned)  (fit in 7.5s)
              precision    recall  f1-score   support

   black_win       0.68      0.55      0.61    101264
   white_win       0.64      0.76      0.69    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.66      0.65    208378



In [8]:
full_vs_pruned = pd.DataFrame(results + pruning_results).set_index('model')[['accuracy', 'f1', 'recall_black_win']]
full_vs_pruned.round(4)

,accuracy,f1,recall_black_win
model,,,
Logistic Regression,0.6536,0.6831,0.5768
Decision Tree (max_depth=8),0.6494,0.6904,0.5316
Random Forest,0.6578,0.6995,0.5343
Gradient Boosting (HistGB),0.6602,0.6951,0.5613
Logistic Regression (pruned),0.6515,0.6871,0.5534
Random Forest (pruned),0.6570,0.6992,0.5316
Gradient Boosting (pruned),0.6575,0.6940,0.5537


## Experiment: split criterion (the closest thing to a "loss function" for trees)
`RandomForestClassifier`'s default `criterion='gini'` picks splits by Gini impurity; `'entropy'` and `'log_loss'` use information-theoretic alternatives instead. This is the one actual lever in this direction — logistic regression's loss (log loss) isn't swappable without changing what "logistic regression" means, and `HistGradientBoostingClassifier` doesn't expose alternative losses for binary classification in sklearn.

In [9]:
criterion_results = []
for criterion in ['gini', 'entropy', 'log_loss']:
    metrics, _ = evaluate_model(
        f'Random Forest (criterion={criterion})',
        RandomForestClassifier(n_estimators=200, max_depth=12, criterion=criterion, n_jobs=-1, random_state=42),
        X_train, X_test,
    )
    criterion_results.append(metrics)

pd.DataFrame(criterion_results).set_index('model')[['accuracy', 'f1', 'recall_black_win', 'fit_seconds']].round(4)

Random Forest (criterion=gini)  (fit in 15.0s)
              precision    recall  f1-score   support

   black_win       0.69      0.53      0.60    101264
   white_win       0.64      0.77      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.66      0.65    208378



Random Forest (criterion=entropy)  (fit in 14.3s)
              precision    recall  f1-score   support

   black_win       0.69      0.53      0.60    101264
   white_win       0.64      0.77      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.66      0.65    208378



Random Forest (criterion=log_loss)  (fit in 14.3s)
              precision    recall  f1-score   support

   black_win       0.69      0.53      0.60    101264
   white_win       0.64      0.77      0.70    107114

    accuracy                           0.66    208378
   macro avg       0.66      0.65      0.65    208378
weighted avg       0.66      0.66      0.65    208378



,accuracy,f1,recall_black_win,fit_seconds
model,,,,
Random Forest (criterion=gini),0.6578,0.6995,0.5343,14.9571
Random Forest (criterion=entropy),0.6579,0.6994,0.5350,14.2970
Random Forest (criterion=log_loss),0.6579,0.6994,0.5350,14.3417
